# Исторический исследовательский код

Эта версия не является основным протоколом статьи. Выводы удалены. Она может требовать исходные локальные файлы и иную схему данных, которые не включены в открытый набор. Для воспроизведения статьи используйте `notebooks/vascular`. Старые метрики не смешиваются с актуальными.


# Интерпретируемая модель госпитальной смертности

Цель — построить понятную многопризнаковую модель риска госпитальной смерти с момента поступления. Используется регуляризованная логистическая регрессия, поэтому влияние факторов представлено в виде скорректированных отношений шансов.

Синтетические наблюдения применяются только для обучения. Проверка качества проводится исключительно на реальных пациентах. В каждом разбиении SMOTENC заново обучается только на обучающей части, поэтому сведения о пациентах из проверочной части не участвуют в создании синтетики.

Модель является исследовательской. Она оценивает прогностические ассоциации, а не причинное влияние факторов, и требует внешней проверки перед клиническим применением.

## 1. Библиотеки и параметры

В модели используются только признаки, известные при поступлении. Взаимозаменяемые признаки не дублируются: категория ФВ используется вместо точного процента, прокси ХБП — вместо СКФ, категория повышенной глюкозы — вместо её точного уровня.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown
from scipy.special import expit, logit
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from imblearn.over_sampling import SMOTENC

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.float_format", lambda value: f"{value:.3f}")

REAL_DATA_PATH = Path("data/hospital_outcomes_prepared.csv")
BALANCED_DATA_PATH = Path("data/hospital_outcomes_balanced_final.csv")
TARGET = "hospital_death"
SAMPLING_STRATEGY = 0.10
BASE_SEED = 2026
OUTER_SPLITS = 5
OUTER_REPEATS = 5
INNER_SPLITS = 3
CANDIDATE_C = [0.03, 0.10, 0.30, 1.00, 3.00]

## 2. Загрузка и контроль данных

In [ ]:
real_data = pd.read_csv(REAL_DATA_PATH)
balanced_data = pd.read_csv(BALANCED_DATA_PATH)

overview = pd.DataFrame(
    {
        "Датасет": ["Реальный", "Итоговый со SMOTENC"],
        "Всего": [len(real_data), len(balanced_data)],
        "Выжили": [int((real_data[TARGET] == 0).sum()), int((balanced_data[TARGET] == 0).sum())],
        "Умерли": [int((real_data[TARGET] == 1).sum()), int((balanced_data[TARGET] == 1).sum())],
        "Синтетические строки": [0, int(balanced_data["is_synthetic"].sum())],
        "Смертность, %": [real_data[TARGET].mean() * 100, balanced_data[TARGET].mean() * 100],
    }
)

display(overview)
display(pd.crosstab(real_data["vascular_group"], real_data[TARGET], margins=True))

## 3. Признаки модели

Числовые показатели после заполнения медианой масштабируются по межквартильному размаху. Поэтому отношение шансов для числового фактора соответствует увеличению примерно на один межквартильный размах. Для каждого числового показателя с пропусками добавляется отдельный индикатор отсутствия данных.

Категории сравниваются с клинически понятной опорной категорией: периферические артерии для сосудистого бассейна, мужской пол, отсутствие соответствующего анамнеза и ФВ выше 50%.

In [ ]:
continuous_generation_features = [
    "age_years",
    "ef_percent",
    "creatinine_value",
    "total_cholesterol",
    "ldl_mmol_l",
    "glucose_value",
    "crp_value",
    "egfr_value",
    "bmi",
]

categorical_generation_features = [
    "vascular_group",
    "female",
    "prior_vascular_surgery",
    "mi_history",
    "stroke_history",
]

continuous_model_features = [
    "age_years",
    "creatinine_value",
    "total_cholesterol",
    "ldl_mmol_l",
    "crp_value",
    "bmi",
]

categorical_model_features = [
    "vascular_group",
    "female",
    "prior_vascular_surgery",
    "ef_category",
    "mi_history",
    "stroke_history",
    "ckd_g3plus_proxy",
    "glucose_elevated_proxy",
]

missing_indicator_features = [
    f"{variable}_missing"
    for variable in continuous_model_features
    if real_data[variable].isna().any()
]

category_levels = {
    "vascular_group": ["peripheral", "absh_aortic_aneurysm", "carotid"],
    "female": ["0", "1"],
    "prior_vascular_surgery": ["0", "1"],
    "ef_category": [">50", "40–50", "<40", "Нет данных"],
    "mi_history": ["0", "1"],
    "stroke_history": ["0", "1"],
    "ckd_g3plus_proxy": ["0", "1", "Нет данных"],
    "glucose_elevated_proxy": ["0", "1", "Нет данных"],
}

for variable in missing_indicator_features:
    category_levels[variable] = ["0", "1"]

feature_labels = {
    "age_years": "Возраст: +1 межквартильный размах",
    "creatinine_value": "Креатинин: +1 межквартильный размах",
    "total_cholesterol": "Общий холестерин: +1 межквартильный размах",
    "ldl_mmol_l": "ЛПНП: +1 межквартильный размах",
    "crp_value": "С-реактивный белок: +1 межквартильный размах",
    "bmi": "Индекс массы тела: +1 межквартильный размах",
    "vascular_group_absh_aortic_aneurysm": "АБШ / аневризма аорты против периферических артерий",
    "vascular_group_carotid": "Сонные артерии против периферических артерий",
    "female_1": "Женский против мужского пола",
    "prior_vascular_surgery_1": "Операция на сосудах в анамнезе: да против нет",
    "ef_category_40–50": "ФВ 40–50% против >50%",
    "ef_category_<40": "ФВ <40% против >50%",
    "ef_category_Нет данных": "ФВ: нет данных против >50%",
    "mi_history_1": "Инфаркт миокарда в анамнезе: да против нет",
    "stroke_history_1": "ОНМК в анамнезе: да против нет",
    "ckd_g3plus_proxy_1": "ХБП G3 и выше против отсутствия",
    "ckd_g3plus_proxy_Нет данных": "ХБП: нет данных против отсутствия",
    "glucose_elevated_proxy_1": "Глюкоза ≥5,6 ммоль/л против <5,6",
    "glucose_elevated_proxy_Нет данных": "Глюкоза: нет данных против <5,6",
}

for variable in missing_indicator_features:
    feature_labels[f"{variable}_1"] = f"{variable.removesuffix('_missing')}: отсутствует значение"

model_feature_table = pd.DataFrame(
    {
        "Тип": ["Числовой"] * len(continuous_model_features)
        + ["Категориальный"] * len(categorical_model_features),
        "Переменная": continuous_model_features + categorical_model_features,
    }
)

display(model_feature_table)

## 4. Функции подготовки и синтеза

Этот блок воспроизводит выбранный ранее вариант балансировки: `SMOTENC`, два ближайших соседа и отношение умерших к выжившим 0,10. Все медианы, масштабы и синтетические наблюдения рассчитываются заново внутри обучающей части.

In [ ]:
def derive_features(frame):
    result = frame.copy()

    result["ef_category"] = pd.Series(pd.NA, index=result.index, dtype="string")
    result.loc[result["ef_percent"] < 40, "ef_category"] = "<40"
    result.loc[result["ef_percent"].between(40, 50, inclusive="both"), "ef_category"] = "40–50"
    result.loc[result["ef_percent"] > 50, "ef_category"] = ">50"

    result["ckd_g3plus_proxy"] = pd.Series(pd.NA, index=result.index, dtype="Int64")
    has_egfr = result["egfr_value"].notna()
    result.loc[has_egfr, "ckd_g3plus_proxy"] = (result.loc[has_egfr, "egfr_value"] < 60).astype(int)

    result["glucose_elevated_proxy"] = pd.Series(pd.NA, index=result.index, dtype="Int64")
    has_glucose = result["glucose_value"].notna()
    result.loc[has_glucose, "glucose_elevated_proxy"] = (result.loc[has_glucose, "glucose_value"] >= 5.6).astype(int)

    return result


def prepare_generation_matrix(frame):
    matrix = frame[continuous_generation_features + categorical_generation_features].copy()
    missing_indicator_columns = []
    indicator_to_variable = {}

    for variable in continuous_generation_features:
        if frame[variable].isna().any():
            indicator = f"{variable}_was_missing"
            missing_indicator_columns.append(indicator)
            indicator_to_variable[indicator] = variable
            matrix[indicator] = frame[variable].isna().astype(int)

            group_sex_median = frame.groupby(["vascular_group", "female"], observed=True)[variable].transform("median")
            group_median = frame.groupby("vascular_group", observed=True)[variable].transform("median")
            matrix[variable] = (
                frame[variable]
                .fillna(group_sex_median)
                .fillna(group_median)
                .fillna(frame[variable].median())
            )

    scaler = RobustScaler()
    matrix[continuous_generation_features] = scaler.fit_transform(matrix[continuous_generation_features])
    categorical_columns = categorical_generation_features + missing_indicator_columns

    for variable in categorical_columns:
        matrix[variable] = matrix[variable].astype("category")

    return matrix, scaler, categorical_columns, missing_indicator_columns, indicator_to_variable


def synthesize_training_data(frame, seed):
    train = frame.reset_index(drop=True).copy()
    matrix, scaler, categorical_columns, missing_columns, indicator_map = prepare_generation_matrix(train)
    sampler = SMOTENC(
        categorical_features=categorical_columns,
        sampling_strategy=SAMPLING_STRATEGY,
        k_neighbors=2,
        random_state=seed,
    )
    resampled_matrix, resampled_target = sampler.fit_resample(matrix, train[TARGET].astype(int))
    synthetic = resampled_matrix.iloc[len(train):].copy()
    synthetic[continuous_generation_features] = scaler.inverse_transform(synthetic[continuous_generation_features])

    for indicator in missing_columns:
        variable = indicator_map[indicator]
        restore_missing = pd.to_numeric(synthetic[indicator], errors="coerce").round().eq(1)
        synthetic.loc[restore_missing, variable] = np.nan

    synthetic = synthetic[continuous_generation_features + categorical_generation_features].copy()
    synthetic["age_years"] = synthetic["age_years"].round()

    for variable in ["female", "prior_vascular_surgery", "mi_history", "stroke_history"]:
        synthetic[variable] = pd.to_numeric(synthetic[variable], errors="coerce").round().astype("Int64")

    synthetic["vascular_group"] = synthetic["vascular_group"].astype("string")
    synthetic = derive_features(synthetic)
    synthetic[TARGET] = 1

    return pd.concat([train, synthetic[train.columns]], ignore_index=True)


def binary_to_text(series):
    numeric = pd.to_numeric(series, errors="coerce").round().astype("Int64")
    return numeric.astype("string").fillna("Нет данных").astype(str)


def make_model_frame(frame):
    result = frame[continuous_model_features].apply(pd.to_numeric, errors="coerce").copy()
    result["vascular_group"] = frame["vascular_group"].astype("string").fillna("Нет данных").astype(str)
    result["ef_category"] = frame["ef_category"].astype("string").fillna("Нет данных").astype(str)

    for variable in [
        "female",
        "prior_vascular_surgery",
        "mi_history",
        "stroke_history",
        "ckd_g3plus_proxy",
        "glucose_elevated_proxy",
    ]:
        result[variable] = binary_to_text(frame[variable])

    for variable in missing_indicator_features:
        source = variable.removesuffix("_missing")
        result[variable] = frame[source].isna().astype(int).astype(str)

    return result


def build_model(c_value):
    categorical_for_model = categorical_model_features + missing_indicator_features
    encoder = OneHotEncoder(
        categories=[category_levels[variable] for variable in categorical_for_model],
        drop="first",
        handle_unknown="ignore",
        sparse_output=False,
    )
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "continuous",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", RobustScaler()),
                    ]
                ),
                continuous_model_features,
            ),
            ("categorical", encoder, categorical_for_model),
        ],
        verbose_feature_names_out=False,
    )
    return Pipeline(
        [
            ("preprocessor", preprocessor),
            (
                "logistic",
                LogisticRegression(
                    penalty="l2",
                    C=c_value,
                    solver="liblinear",
                    max_iter=3000,
                    random_state=BASE_SEED,
                ),
            ),
        ]
    )


def prior_correct(probabilities, real_prevalence, model_prevalence):
    clipped = np.clip(np.asarray(probabilities, dtype=float), 1e-6, 1 - 1e-6)
    shift = logit(real_prevalence) - logit(model_prevalence)
    return expit(logit(clipped) + shift)

## 5. Вложенная перекрёстная проверка на реальных пациентах

Внешний цикл оценивает качество на пациентах, которых модель не видела. Внутренний цикл выбирает силу L2-регуляризации по PR-AUC. Такой показатель особенно важен при редком исходе. Каждая проверочная часть содержит только реальные наблюдения.

In [ ]:
X_real = real_data.reset_index(drop=True).copy()
y_real = X_real[TARGET].astype(int).to_numpy()

outer_cv = RepeatedStratifiedKFold(
    n_splits=OUTER_SPLITS,
    n_repeats=OUTER_REPEATS,
    random_state=BASE_SEED,
)

prediction_sum = np.zeros(len(X_real), dtype=float)
prediction_count = np.zeros(len(X_real), dtype=int)
fold_rows = []
coefficient_rows = []

for fold_id, (train_index, test_index) in enumerate(outer_cv.split(X_real, y_real), start=1):
    outer_train = X_real.iloc[train_index].reset_index(drop=True)
    outer_test = X_real.iloc[test_index].reset_index(drop=True)
    inner_cv = StratifiedKFold(
        n_splits=INNER_SPLITS,
        shuffle=True,
        random_state=BASE_SEED + fold_id,
    )
    c_scores = []

    for c_value in CANDIDATE_C:
        inner_scores = []
        inner_target = outer_train[TARGET].astype(int).to_numpy()

        for inner_id, (inner_train_index, inner_valid_index) in enumerate(
            inner_cv.split(outer_train, inner_target),
            start=1,
        ):
            inner_train = outer_train.iloc[inner_train_index].reset_index(drop=True)
            inner_valid = outer_train.iloc[inner_valid_index].reset_index(drop=True)
            inner_balanced = synthesize_training_data(
                inner_train,
                BASE_SEED + fold_id * 100 + inner_id,
            )
            model = build_model(c_value)
            model.fit(make_model_frame(inner_balanced), inner_balanced[TARGET].astype(int))
            raw_probability = model.predict_proba(make_model_frame(inner_valid))[:, 1]
            corrected_probability = prior_correct(
                raw_probability,
                inner_train[TARGET].mean(),
                inner_balanced[TARGET].mean(),
            )
            inner_scores.append(average_precision_score(inner_valid[TARGET], corrected_probability))

        c_scores.append((c_value, float(np.mean(inner_scores))))

    selected_c, selected_inner_score = max(c_scores, key=lambda item: (item[1], -item[0]))
    outer_balanced = synthesize_training_data(outer_train, BASE_SEED + fold_id * 1000)
    final_fold_model = build_model(selected_c)
    final_fold_model.fit(make_model_frame(outer_balanced), outer_balanced[TARGET].astype(int))
    raw_probability = final_fold_model.predict_proba(make_model_frame(outer_test))[:, 1]
    corrected_probability = prior_correct(
        raw_probability,
        outer_train[TARGET].mean(),
        outer_balanced[TARGET].mean(),
    )

    prediction_sum[test_index] += corrected_probability
    prediction_count[test_index] += 1

    fold_rows.append(
        {
            "fold": fold_id,
            "selected_C": selected_c,
            "inner_PR_AUC": selected_inner_score,
            "test_ROC_AUC": roc_auc_score(outer_test[TARGET], corrected_probability),
            "test_PR_AUC": average_precision_score(outer_test[TARGET], corrected_probability),
            "test_Brier": brier_score_loss(outer_test[TARGET], corrected_probability),
            "real_train_n": len(outer_train),
            "synthetic_train_n": len(outer_balanced) - len(outer_train),
            "real_test_n": len(outer_test),
        }
    )

    feature_names = final_fold_model.named_steps["preprocessor"].get_feature_names_out()
    coefficients = final_fold_model.named_steps["logistic"].coef_[0]

    for feature_name, coefficient in zip(feature_names, coefficients):
        coefficient_rows.append(
            {
                "fold": fold_id,
                "feature": feature_name,
                "coefficient": coefficient,
            }
        )

oof_probability = prediction_sum / prediction_count
fold_results = pd.DataFrame(fold_rows)
coefficient_results = pd.DataFrame(coefficient_rows)

display(
    fold_results["selected_C"]
    .value_counts()
    .sort_index()
    .rename_axis("C")
    .rename("Число внешних разбиений")
    .to_frame()
)

display(fold_results.describe().T)

## 6. Итоговые показатели качества

Для каждого пациента усредняются прогнозы из повторных внешних разбиений. Доверительные интервалы получены стратифицированным бутстрэпом реальных пациентов. Они описывают внутреннюю устойчивость, но не заменяют внешнюю валидацию.

In [ ]:
def metric_values(target, probability):
    return {
        "ROC-AUC": roc_auc_score(target, probability),
        "PR-AUC": average_precision_score(target, probability),
        "Brier score": brier_score_loss(target, probability),
    }


def bootstrap_metrics(target, probability, n_bootstrap=1000, seed=BASE_SEED):
    rng = np.random.default_rng(seed)
    target = np.asarray(target)
    probability = np.asarray(probability)
    alive_index = np.flatnonzero(target == 0)
    deceased_index = np.flatnonzero(target == 1)
    rows = []

    for _ in range(n_bootstrap):
        sample_index = np.concatenate(
            [
                rng.choice(alive_index, size=len(alive_index), replace=True),
                rng.choice(deceased_index, size=len(deceased_index), replace=True),
            ]
        )
        rows.append(metric_values(target[sample_index], probability[sample_index]))

    return pd.DataFrame(rows)


point_metrics = metric_values(y_real, oof_probability)
bootstrap_result = bootstrap_metrics(y_real, oof_probability)
metric_table = pd.DataFrame(
    [
        {
            "Метрика": metric,
            "Оценка": value,
            "95% ДИ, нижняя": bootstrap_result[metric].quantile(0.025),
            "95% ДИ, верхняя": bootstrap_result[metric].quantile(0.975),
        }
        for metric, value in point_metrics.items()
    ]
)

display(metric_table)

fpr, tpr, roc_thresholds = roc_curve(y_real, oof_probability)
finite_thresholds = np.isfinite(roc_thresholds)
youden_index = np.argmax((tpr - fpr)[finite_thresholds])
selected_threshold = roc_thresholds[finite_thresholds][youden_index]
predicted_class = (oof_probability >= selected_threshold).astype(int)
matrix = confusion_matrix(y_real, predicted_class)
tn, fp, fn, tp = matrix.ravel()

classification_table = pd.DataFrame(
    {
        "Показатель": [
            "Исследовательский порог риска",
            "Чувствительность",
            "Специфичность",
            "Положительная прогностическая ценность",
            "Отрицательная прогностическая ценность",
            "Сбалансированная точность",
        ],
        "Значение": [
            selected_threshold,
            recall_score(y_real, predicted_class),
            tn / (tn + fp),
            precision_score(y_real, predicted_class, zero_division=0),
            tn / (tn + fn),
            balanced_accuracy_score(y_real, predicted_class),
        ],
    }
)

display(classification_table)

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].plot(fpr, tpr, color="#355C7D", linewidth=2, label=f"ROC-AUC = {point_metrics['ROC-AUC']:.3f}")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="grey")
axes[0].set_xlabel("1 − специфичность")
axes[0].set_ylabel("Чувствительность")
axes[0].set_title("ROC-кривая")
axes[0].legend(loc="lower right")

precision, recall, _ = precision_recall_curve(y_real, oof_probability)
axes[1].plot(recall, precision, color="#C06C84", linewidth=2, label=f"PR-AUC = {point_metrics['PR-AUC']:.3f}")
axes[1].axhline(y_real.mean(), linestyle="--", color="grey", label=f"Базовая частота = {y_real.mean():.3f}")
axes[1].set_xlabel("Чувствительность")
axes[1].set_ylabel("Положительная прогностическая ценность")
axes[1].set_title("Precision–Recall кривая")
axes[1].legend(loc="upper right")

sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Прогноз: жив", "Прогноз: смерть"],
    yticklabels=["Факт: жив", "Факт: смерть"],
    ax=axes[2],
)
axes[2].set_title(f"Матрица ошибок при пороге {selected_threshold:.3f}")
axes[2].set_xlabel("")
axes[2].set_ylabel("")

figure.tight_layout()
plt.show()
plt.close(figure)

## 7. Калибровка и распределение риска

Калибровка показывает, соответствует ли прогнозируемая вероятность фактической доле смертей. Из-за малого числа событий график следует трактовать осторожно. Коррекция базовой частоты возвращает прогнозы от обучающей доли смертей после SMOTENC к реальной частоте исхода.

In [ ]:
observed_rate, predicted_rate = calibration_curve(
    y_real,
    oof_probability,
    n_bins=5,
    strategy="quantile",
)

risk_frame = pd.DataFrame(
    {
        "Госпитальный исход": np.where(y_real == 1, "Умер", "Выжил"),
        "Прогнозируемый риск": oof_probability,
    }
)

figure, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot([0, 1], [0, 1], linestyle="--", color="grey", label="Идеальная калибровка")
axes[0].plot(predicted_rate, observed_rate, marker="o", color="#6C9A8B", linewidth=2, label="Модель")
axes[0].set_xlim(0, max(0.25, predicted_rate.max() * 1.15))
axes[0].set_ylim(0, max(0.25, observed_rate.max() * 1.15))
axes[0].set_xlabel("Средний прогнозируемый риск")
axes[0].set_ylabel("Наблюдаемая смертность")
axes[0].set_title("Калибровочная кривая")
axes[0].legend()

sns.boxplot(
    data=risk_frame,
    x="Госпитальный исход",
    y="Прогнозируемый риск",
    hue="Госпитальный исход",
    palette={"Выжил": "#355C7D", "Умер": "#C06C84"},
    legend=False,
    ax=axes[1],
)
axes[1].set_xlabel("")
axes[1].set_ylabel("Внутренне проверенный риск")
axes[1].set_title("Распределение риска по исходу")

figure.tight_layout()
plt.show()
plt.close(figure)

## 8. Интерпретация факторов

Отношение шансов выше 1 соответствует увеличению прогнозируемых шансов госпитальной смерти, ниже 1 — уменьшению. Интервалы на графике — 10-й и 90-й процентили коэффициентов во внешних разбиениях, то есть диапазон устойчивости модели, а не классический доверительный интервал.

Синтетические пациенты не являются новыми независимыми клиническими наблюдениями. Поэтому величина коэффициента и его устойчивость важнее формального p-value, который для этой модели не рассчитывается.

In [ ]:
selected_final_c = float(fold_results["selected_C"].median())
final_model = build_model(selected_final_c)
final_model.fit(make_model_frame(balanced_data), balanced_data[TARGET].astype(int))

final_feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
final_coefficients = final_model.named_steps["logistic"].coef_[0]
final_coefficient_map = dict(zip(final_feature_names, final_coefficients))

coefficient_summary = (
    coefficient_results
    .groupby("feature")["coefficient"]
    .agg(
        median_beta="median",
        beta_p10=lambda values: values.quantile(0.10),
        beta_p90=lambda values: values.quantile(0.90),
        positive_share=lambda values: (values > 0).mean(),
    )
    .reset_index()
)

coefficient_summary["final_beta"] = coefficient_summary["feature"].map(final_coefficient_map)
coefficient_summary["adjusted_OR"] = np.exp(coefficient_summary["final_beta"])
coefficient_summary["stability_OR"] = np.exp(coefficient_summary["median_beta"])
coefficient_summary["stability_OR_p10"] = np.exp(coefficient_summary["beta_p10"])
coefficient_summary["stability_OR_p90"] = np.exp(coefficient_summary["beta_p90"])
coefficient_summary["sign_stability"] = coefficient_summary["positive_share"].where(
    coefficient_summary["positive_share"] >= 0.5,
    1 - coefficient_summary["positive_share"],
)
coefficient_summary["label_ru"] = coefficient_summary["feature"].map(feature_labels).fillna(coefficient_summary["feature"])
coefficient_summary["is_missingness_term"] = coefficient_summary["feature"].str.contains("missing|Нет данных")

clinical_effects = (
    coefficient_summary.loc[~coefficient_summary["is_missingness_term"]]
    .sort_values("stability_OR")
    .reset_index(drop=True)
)

display(
    clinical_effects[
        [
            "label_ru",
            "adjusted_OR",
            "stability_OR",
            "stability_OR_p10",
            "stability_OR_p90",
            "sign_stability",
        ]
    ].rename(
        columns={
            "label_ru": "Фактор",
            "adjusted_OR": "ОШ итоговой модели",
            "stability_OR": "Медианное ОШ по разбиениям",
            "stability_OR_p10": "10-й процентиль",
            "stability_OR_p90": "90-й процентиль",
            "sign_stability": "Доля разбиений с одинаковым направлением",
        }
    )
)

In [ ]:
plot_data = clinical_effects.copy()
y_positions = np.arange(len(plot_data))
x_values = plot_data["stability_OR"].to_numpy()
x_errors = np.vstack(
    [
        x_values - plot_data["stability_OR_p10"].to_numpy(),
        plot_data["stability_OR_p90"].to_numpy() - x_values,
    ]
)

figure, axis = plt.subplots(figsize=(11, max(6, len(plot_data) * 0.48)))
axis.errorbar(
    x_values,
    y_positions,
    xerr=x_errors,
    fmt="o",
    color="#355C7D",
    ecolor="#6C9A8B",
    capsize=3,
)
axis.axvline(1, color="grey", linestyle="--")
axis.set_xscale("log")
axis.set_yticks(y_positions)
axis.set_yticklabels(plot_data["label_ru"])
axis.set_xlabel("Скорректированное отношение шансов, логарифмическая шкала")
axis.set_title("Направление и устойчивость эффектов логистической модели")
figure.tight_layout()
plt.show()
plt.close(figure)

## 9. Роль отсутствующих данных

Этот технический блок показывает, использует ли модель само отсутствие измерения как прогностический сигнал. Такие эффекты не следует трактовать как биологические факторы риска: они могут отражать особенности обследования и заполнения медицинской документации.

In [ ]:
missingness_effects = coefficient_summary.loc[
    coefficient_summary["is_missingness_term"],
    ["label_ru", "adjusted_OR", "stability_OR", "stability_OR_p10", "stability_OR_p90", "sign_stability"],
].sort_values("stability_OR", ascending=False)

display(
    missingness_effects.rename(
        columns={
            "label_ru": "Технический признак",
            "adjusted_OR": "ОШ итоговой модели",
            "stability_OR": "Медианное ОШ по разбиениям",
            "stability_OR_p10": "10-й процентиль",
            "stability_OR_p90": "90-й процентиль",
            "sign_stability": "Доля разбиений с одинаковым направлением",
        }
    )
)

## 10. Автоматическое резюме

In [ ]:
top_risk = clinical_effects.sort_values("stability_OR", ascending=False).head(5)
top_protective = clinical_effects.sort_values("stability_OR", ascending=True).head(5)
stable_effects = clinical_effects.loc[clinical_effects["sign_stability"] >= 0.80]

display(Markdown(
    f'''
**Внутренняя проверка на реальных пациентах**

- ROC-AUC: **{point_metrics['ROC-AUC']:.3f}**.
- PR-AUC: **{point_metrics['PR-AUC']:.3f}** при исходной частоте смерти **{y_real.mean():.3f}**.
- Brier score: **{point_metrics['Brier score']:.3f}**.
- Для итоговой модели выбрано `C = {selected_final_c:.2f}`.
- У {len(stable_effects)} из {len(clinical_effects)} клинических коэффициентов направление совпадало не менее чем в 80% внешних разбиений.

Результаты следует использовать для выделения кандидатов на дальнейшую проверку. Синтетика улучшает обучаемость модели, но не увеличивает число независимых смертельных исходов и не заменяет внешнюю клиническую валидацию.
'''
))

display(Markdown("**Наиболее выраженное повышение шансов по медианному коэффициенту**"))
display(
    top_risk[["label_ru", "stability_OR", "sign_stability"]].rename(
        columns={
            "label_ru": "Фактор",
            "stability_OR": "Медианное ОШ",
            "sign_stability": "Устойчивость направления",
        }
    )
)

display(Markdown("**Наиболее выраженное снижение шансов по медианному коэффициенту**"))
display(
    top_protective[["label_ru", "stability_OR", "sign_stability"]].rename(
        columns={
            "label_ru": "Фактор",
            "stability_OR": "Медианное ОШ",
            "sign_stability": "Устойчивость направления",
        }
    )
)